In [1]:
import networkx as nx
import pandas as pd

# First, let's make sure we have the graph loaded
# If you already have G defined, you can skip this part
try:
    # Check if G exists
    G.number_of_nodes()
except NameError:
    # If G doesn't exist, load it again
    import json
    with open("../../textlab/texts/Others/wil_texter_2.json", "r", encoding="utf-8-sig") as f:
        data = json.load(f)
    
    G = nx.Graph()
    
    # Add nodes from the items
    for item in data["network"]["items"]:
        node_id = str(item["id"])
        label = item["label"]
        
        attrs = {
            "label": label,
            "x": item.get("x", 0),
            "y": item.get("y", 0),
            "cluster": item.get("cluster", 0)
        }
        
        if "weights" in item:
            for weight_name, weight_value in item["weights"].items():
                attr_name = f"weight_{weight_name.replace(' ', '_')}"
                attrs[attr_name] = weight_value
        
        if "scores" in item:
            for score_name, score_value in item["scores"].items():
                attr_name = f"score_{score_name.replace(' ', '_')}"
                attrs[attr_name] = score_value
        
        G.add_node(node_id, **attrs)
    
    # Add edges from the links
    if "links" in data["network"]:
        for link in data["network"]["links"]:
            source = str(link["source_id"])
            target = str(link["target_id"])
            weight = link.get("strength", 1.0)
            G.add_edge(source, target, weight=weight)
    
    print(f"Graph created with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

# Calculate all three centrality measures
betweenness = nx.betweenness_centrality(G)
closeness = nx.closeness_centrality(G)
eigenvector = nx.eigenvector_centrality(G, max_iter=1000)

# Function to get top nodes by centrality
def get_top_nodes(centrality_dict, n=5):
    return sorted(centrality_dict.items(), key=lambda x: x[1], reverse=True)[:n]

# Get top 5 nodes for each centrality measure
top_betweenness = get_top_nodes(betweenness)
top_closeness = get_top_nodes(closeness)
top_eigenvector = get_top_nodes(eigenvector)

# Print results
print("Top 5 nodes by Betweenness Centrality:")
for node_id, score in top_betweenness:
    label = G.nodes[node_id]["label"]
    print(f"  {label}: {score:.4f}")

print("\nTop 5 nodes by Closeness Centrality:")
for node_id, score in top_closeness:
    label = G.nodes[node_id]["label"]
    print(f"  {label}: {score:.4f}")

print("\nTop 5 nodes by Eigenvector Centrality:")
for node_id, score in top_eigenvector:
    label = G.nodes[node_id]["label"]
    print(f"  {label}: {score:.4f}")

# Create a DataFrame with all centrality measures and node attributes
centrality_data = []
for node_id in G.nodes():
    node_data = G.nodes[node_id]
    centrality_data.append({
        "label": node_data["label"],
        "cluster": node_data.get("cluster", 0),
        "betweenness": betweenness[node_id],
        "closeness": closeness[node_id],
        "eigenvector": eigenvector[node_id],
        "documents": node_data.get("weight_Documents", 0),
        "total_link_strength": node_data.get("weight_Total_link_strength", 0)
    })

df = pd.DataFrame(centrality_data)

# Sort by different centrality measures
print("\nNodes sorted by betweenness centrality:")
print(df.sort_values("betweenness", ascending=False).head()[["label", "cluster", "betweenness"]])

print("\nNodes sorted by closeness centrality:")
print(df.sort_values("closeness", ascending=False).head()[["label", "cluster", "closeness"]])

print("\nNodes sorted by eigenvector centrality:")
print(df.sort_values("eigenvector", ascending=False).head()[["label", "cluster", "eigenvector"]])

# Correlation between centrality measures and other attributes
print("\nCorrelation between centrality measures and document count:")
print(df[["betweenness", "closeness", "eigenvector", "documents", "total_link_strength"]].corr())

Graph created with 105 nodes and 360 edges
Top 5 nodes by Betweenness Centrality:
  lundh snis, ulrika: 0.2858
  johansson, kristina: 0.1601
  vallo hult, helena: 0.1127
  svensson, ann: 0.1100
  svensson, lars: 0.1032

Top 5 nodes by Closeness Centrality:
  lundh snis, ulrika: 0.5605
  vallo hult, helena: 0.4858
  gellerstedt, martin: 0.4637
  pennbrant, sandra: 0.4637
  svensson, lars: 0.4595

Top 5 nodes by Eigenvector Centrality:
  lundh snis, ulrika: 0.3926
  vallo hult, helena: 0.2656
  svensson, lars: 0.2547
  norström, livia: 0.2295
  gellerstedt, martin: 0.2263

Nodes sorted by betweenness centrality:
                  label  cluster  betweenness
58   lundh snis, ulrika        1     0.285803
48  johansson, kristina        1     0.160117
96   vallo hult, helena        1     0.112746
90        svensson, ann        8     0.110023
91       svensson, lars        1     0.103244

Nodes sorted by closeness centrality:
                  label  cluster  closeness
58   lundh snis, ulrika

In [2]:

# Basic graph information
print(f"Number of nodes: {G.number_of_nodes()}")
print(f"Number of edges: {G.number_of_edges()}")
print(f"Graph type: {'Directed' if G.is_directed() else 'Undirected'}")
print(f"Graph density: {nx.density(G):.4f}")

Number of nodes: 105
Number of edges: 360
Graph type: Undirected
Graph density: 0.0659


In [3]:
# Connectivity measures
print(f"Number of connected components: {nx.number_connected_components(G)}")
print(f"Size of largest component: {max(len(comp) for comp in nx.connected_components(G))}")

# Only calculate path-based metrics if the graph is connected
if nx.is_connected(G):
    print(f"Average shortest path length: {nx.average_shortest_path_length(G):.4f}")
    print(f"Diameter: {nx.diameter(G)}")
else:
    print("Graph is not connected - path-based metrics not applicable")

Number of connected components: 2
Size of largest component: 104
Graph is not connected - path-based metrics not applicable


In [4]:
# Clustering coefficient
print(f"Average clustering coefficient: {nx.average_clustering(G):.4f}")

# Transitivity (global clustering)
print(f"Transitivity: {nx.transitivity(G):.4f}")

Average clustering coefficient: 0.4958
Transitivity: 0.3396


In [5]:
# Different centrality measures
degree_centrality = nx.degree_centrality(G)
betweenness_centrality = nx.betweenness_centrality(G)
closeness_centrality = nx.closeness_centrality(G)
eigenvector_centrality = nx.eigenvector_centrality(G)

# Find nodes with highest values for each centrality
print("\nTop nodes by centrality measures:")
print(f"Degree centrality: {max(degree_centrality, key=degree_centrality.get)}")
print(f"Betweenness centrality: {max(betweenness_centrality, key=betweenness_centrality.get)}")
print(f"Closeness centrality: {max(closeness_centrality, key=closeness_centrality.get)}")
print(f"Eigenvector centrality: {max(eigenvector_centrality, key=eigenvector_centrality.get)}")


Top nodes by centrality measures:
Degree centrality: 543
Betweenness centrality: 543
Closeness centrality: 543
Eigenvector centrality: 543


In [ ]:
# Degree statistics
degrees = [G.degree(n) for n in G.nodes()]
print(f"\nDegree statistics:")
print(f"Average degree: {sum(degrees)/len(degrees):.2f}")
print(f"Minimum degree: {min(degrees)}")
print(f"Maximum degree: {max(degrees)}")


Degree statistics:
Average degree: 6.86
Minimum degree: 0
Maximum degree: 42
